# Preprocessing Lokal – Dataset Klasifikasi Sampah (SigLIP)

**Alur:**
1. Tahap 1 – Deduplikasi Gambar (pixel hash SHA256 + MD5 file)
2. Tahap 2 – Filter Kualitas & Resolusi + Konversi ke RGB
3. Tahap 3 – Hitung Class Weights & Catat Rencana Augmentasi
4. Tahap 4 – Stratified Split train/val/test (70/15/15)

**Deliverable:**
- `dedup_log.csv` – daftar duplikat yang dipindahkan
- `metadata_kualitas.csv` – metadata gambar + flag kualitas
- `class_weights.json` – bobot kelas untuk loss function
- `train.csv`, `val.csv`, `test.csv` – manifest split

In [1]:
# ============================================================
# KONFIGURASI – Ganti ROOT_DIR sesuai lokasi folder dataset
# ============================================================
import os

ROOT_DIR = r"d:\UGM\Semester 5\Senpro\Project\Senpro"   # root project
DATASET_DIR = os.path.join(ROOT_DIR, "Dataset")           # folder berisi 0_Recylable, 1_Organik, 2_B3
OUTPUT_DIR = os.path.join(ROOT_DIR, "preprocessing_output") # hasil output semua file CSV/JSON

# Label mapping: nama folder -> nama kelas
CLASS_MAP = {
    "0_Recylable": "Recylable",
    "1_Organik": "Organik",
    "2_B3": "B3",
}

# Ambang resolusi minimum
MIN_RESOLUTION = 50   # piksel (lebar DAN tinggi harus >= nilai ini)

# Threshold deteksi kecerahan
DARK_THRESHOLD   = 50   # rata-rata brightness (0-255) di bawah ini → terindikasi gelap
BRIGHT_THRESHOLD = 220  # rata-rata brightness di atas ini → terindikasi overexposed

# Split ratio
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] OUTPUT_DIR: {OUTPUT_DIR}")
print(f"[OK] DATASET_DIR: {DATASET_DIR}")

[OK] OUTPUT_DIR: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output
[OK] DATASET_DIR: d:\UGM\Semester 5\Senpro\Project\Senpro\Dataset


In [2]:
# ============================================================
# INSTALASI DEPENDENSI (jalankan sekali jika belum ada)
# ============================================================
# %pip install tqdm pillow pandas scikit-learn

In [3]:
# ============================================================
# IMPORTS
# ============================================================
import hashlib
import json
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print("[OK] Semua library berhasil diimport.")

[OK] Semua library berhasil diimport.


---
## TAHAP 1 – Deduplikasi Gambar

In [4]:
def compute_md5(filepath: Path) -> str:
    """Hitung MD5 hash dari byte file mentah."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


def compute_pixel_sha256(filepath: Path) -> str | None:
    """
    Hitung SHA256 dari kombinasi (lebar, tinggi, pixel_bytes_RGB).
    Return None jika gambar tidak bisa dibuka.
    """
    try:
        with Image.open(filepath) as img:
            img_rgb = img.convert('RGB')
            w, h = img_rgb.size
            pixel_bytes = img_rgb.tobytes()
        h_sha = hashlib.sha256()
        h_sha.update(w.to_bytes(4, 'little'))
        h_sha.update(h.to_bytes(4, 'little'))
        h_sha.update(pixel_bytes)
        return h_sha.hexdigest()
    except Exception:
        return None


def collect_all_images(dataset_dir: str, class_map: dict) -> list[dict]:
    """
    Kumpulkan semua file gambar dari seluruh subfolder (rekursif).
    Return list of dict {path, rel_path, label, filename}.
    """
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif', '.tiff'}
    records = []
    for folder_name, label in class_map.items():
        class_dir = Path(dataset_dir) / folder_name
        if not class_dir.exists():
            print(f"[WARN] Folder tidak ditemukan: {class_dir}")
            continue
        # rglob → rekursif ke semua subfolder
        for fpath in sorted(class_dir.rglob('*')):
            if fpath.suffix.lower() in valid_ext and fpath.is_file():
                rel_path = fpath.relative_to(dataset_dir)
                records.append({
                    'path': fpath,
                    'rel_path': str(rel_path),
                    'label': label,
                    'filename': fpath.name,
                    'class_folder': folder_name,
                })
    return records


print("[INFO] Mengumpulkan semua gambar...")
all_images = collect_all_images(DATASET_DIR, CLASS_MAP)
print(f"[OK] Total gambar ditemukan: {len(all_images)}")

# Ringkasan per kelas
from collections import Counter
label_counts = Counter(r['label'] for r in all_images)
for label, cnt in sorted(label_counts.items()):
    print(f"  {label}: {cnt} gambar")

[INFO] Mengumpulkan semua gambar...


[OK] Total gambar ditemukan: 30325
  B3: 3637 gambar
  Organik: 13744 gambar
  Recylable: 12944 gambar


In [5]:
# Hitung hash untuk setiap gambar
print("[INFO] Menghitung signature gambar (MD5 + SHA256 pixel)...")
for rec in tqdm(all_images, desc="Hashing", unit="img"):
    rec['md5']         = compute_md5(rec['path'])
    rec['pixel_sha256'] = compute_pixel_sha256(rec['path'])

print("[OK] Selesai hashing.")

[INFO] Menghitung signature gambar (MD5 + SHA256 pixel)...


Hashing:   0%|          | 0/30325 [00:00<?, ?img/s]

Hashing:   0%|          | 9/30325 [00:00<05:38, 89.43img/s]

Hashing:   0%|          | 25/30325 [00:00<03:57, 127.37img/s]

Hashing:   0%|          | 42/30325 [00:00<03:29, 144.82img/s]

...

Hashing: 100%|█████████▉| 30217/30325 [02:38<00:00, 351.46img/s]

Hashing: 100%|█████████▉| 30272/30325 [02:38<00:00, 335.51img/s]

Hashing: 100%|█████████▉| 30323/30325 [02:38<00:00, 363.96img/s]

Hashing: 100%|██████████| 30325/30325 [02:38<00:00, 191.15img/s]

[OK] Selesai hashing.


In [6]:
def find_duplicates(records: list[dict]) -> tuple[set, list[dict]]:
    """
    Deteksi duplikat berdasarkan union MD5 dan pixel_sha256.
    Untuk setiap grup duplikat, pilih representatif terbaik:
      - Prioritas: resolusi tertinggi (lebar * tinggi)
      - Tie-break: nama file terpendek, lalu alfabetis
    Return:
      - keep_paths: set of path string untuk file yang DIPERTAHANKAN
      - dup_log: list of dict untuk file yang DIHAPUS (dipindahkan)
    """
    # Kelompokkan berdasarkan masing-masing hash
    md5_groups   = defaultdict(list)
    sha256_groups = defaultdict(list)

    for rec in records:
        if rec['md5']:
            md5_groups[rec['md5']].append(rec)
        if rec['pixel_sha256']:
            sha256_groups[rec['pixel_sha256']].append(rec)

    # Kumpulkan semua grup duplikat (union)
    # Gunakan path sebagai identifier
    dup_groups: list[list[dict]] = []
    
    def get_resolution(rec):
        try:
            with Image.open(rec['path']) as img:
                return img.width * img.height
        except Exception:
            return 0

    def pick_representative(group: list[dict]) -> dict:
        """Pilih gambar dengan resolusi terbesar; tie-break: nama file pendek lalu alfa."""
        return max(
            group,
            key=lambda r: (get_resolution(r), -len(r['filename']), r['filename'])
        )

    # Proses MD5 duplikat
    md5_dup_paths: set[str] = set()  # path yang sudah masuk grup
    for h, group in md5_groups.items():
        if len(group) < 2:
            continue
        dup_groups.append(group)
        for r in group:
            md5_dup_paths.add(str(r['path']))

    # Proses SHA256 duplikat (tambahkan grup baru yang belum tercakup MD5)
    sha256_dup_paths: set[str] = set()
    for h, group in sha256_groups.items():
        if len(group) < 2:
            continue
        group_paths = {str(r['path']) for r in group}
        # Hanya tambahkan jika ada path yang belum tercakup MD5 group
        if not group_paths.issubset(md5_dup_paths):
            dup_groups.append(group)
            for p in group_paths:
                sha256_dup_paths.add(p)

    # Tentukan yang dipertahankan per grup, sisanya → hapus/pindahkan
    path_to_rec = {str(r['path']): r for r in records}
    to_remove: set[str] = set()
    dup_log: list[dict] = []

    for group in dup_groups:
        rep = pick_representative(group)
        for r in group:
            if str(r['path']) != str(rep['path']) and str(r['path']) not in to_remove:
                to_remove.add(str(r['path']))
                dup_log.append({
                    'file_dihapus': r['rel_path'],
                    'label': r['label'],
                    'file_representatif': rep['rel_path'],
                    'md5': r['md5'],
                    'pixel_sha256': r['pixel_sha256'] or '',
                    'alasan': 'md5_match' if r['md5'] == rep['md5'] else 'pixel_match',
                })

    keep_paths = {str(r['path']) for r in records} - to_remove
    return keep_paths, dup_log


print("[INFO] Mendeteksi duplikat...")
keep_paths, dup_log = find_duplicates(all_images)
print(f"[OK] Duplikat ditemukan: {len(dup_log)} file")
print(f"[OK] File dipertahankan: {len(keep_paths)} file")

[INFO] Mendeteksi duplikat...
[OK] Duplikat ditemukan: 0 file
[OK] File dipertahankan: 30325 file


In [7]:
# Pindahkan duplikat ke folder duplicates_removed
DUPLICATES_DIR = Path(DATASET_DIR) / "duplicates_removed"
DUPLICATES_DIR.mkdir(exist_ok=True)

print(f"[INFO] Memindahkan {len(dup_log)} duplikat ke: {DUPLICATES_DIR}")
move_errors = []
for entry in tqdm(dup_log, desc="Memindahkan duplikat", unit="file"):
    src = Path(DATASET_DIR) / entry['file_dihapus']
    # Pertahankan struktur subfolder di dalam duplicates_removed
    dst = DUPLICATES_DIR / entry['file_dihapus']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        try:
            shutil.move(str(src), str(dst))
        except Exception as e:
            move_errors.append({'file': entry['file_dihapus'], 'error': str(e)})
    else:
        move_errors.append({'file': entry['file_dihapus'], 'error': 'source_not_found'})

if move_errors:
    print(f"[WARN] {len(move_errors)} file gagal dipindahkan.")
else:
    print("[OK] Semua duplikat berhasil dipindahkan.")

[INFO] Memindahkan 0 duplikat ke: d:\UGM\Semester 5\Senpro\Project\Senpro\Dataset\duplicates_removed


Memindahkan duplikat: 0file [00:00, ?file/s]

Memindahkan duplikat: 0file [00:00, ?file/s]

[OK] Semua duplikat berhasil dipindahkan.


In [8]:
# Simpan dedup_log.csv
dedup_log_path = os.path.join(OUTPUT_DIR, "dedup_log.csv")
df_dedup = pd.DataFrame(dup_log)
df_dedup.to_csv(dedup_log_path, index=False)
print(f"[OK] dedup_log.csv disimpan: {dedup_log_path}")

# Laporan per kelas
print("\n=== LAPORAN DEDUPLIKASI ===")
before_counts = Counter(r['label'] for r in all_images)
kept_records  = [r for r in all_images if str(r['path']) in keep_paths]
after_counts  = Counter(r['label'] for r in kept_records)
for label in sorted(before_counts.keys()):
    b = before_counts[label]
    a = after_counts.get(label, 0)
    print(f"  {label}: {b} → {a} (dihapus: {b - a})")
print(f"  TOTAL: {sum(before_counts.values())} → {sum(after_counts.values())}")

[OK] dedup_log.csv disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\dedup_log.csv

=== LAPORAN DEDUPLIKASI ===
  B3: 3637 → 3637 (dihapus: 0)
  Organik: 13744 → 13744 (dihapus: 0)
  Recylable: 12944 → 12944 (dihapus: 0)
  TOTAL: 30325 → 30325


---
## TAHAP 2 – Filter Kualitas & Resolusi + Konversi RGB

In [9]:
def analyze_image_quality(filepath: Path, dark_thresh: int, bright_thresh: int) -> dict:
    """
    Analisis kualitas gambar: status, resolusi, mode warna, flag kecerahan.
    Jika gambar bukan RGB, konversi dan simpan ulang.
    Return dict dengan hasil analisis.
    """
    result = {
        'status': 'ok',
        'lebar': None,
        'tinggi': None,
        'mode_awal': None,
        'dikonversi_ke_rgb': False,
        'terindikasi_gelap': False,
        'terindikasi_overexposed': False,
        'rata_kecerahan': None,
        'error': '',
    }
    try:
        with Image.open(filepath) as img:
            result['mode_awal'] = img.mode
            result['lebar']    = img.width
            result['tinggi']   = img.height

            # Filter resolusi minimum
            if img.width < MIN_RESOLUTION or img.height < MIN_RESOLUTION:
                result['status'] = 'dropped_low_res'
                return result

            # Konversi ke RGB jika perlu
            if img.mode != 'RGB':
                img_rgb = img.convert('RGB')
                result['dikonversi_ke_rgb'] = True
                # Simpan ulang menimpa file asli dengan format yang sama
                suffix = filepath.suffix.lower()
                fmt_map = {'.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG',
                           '.bmp': 'BMP', '.webp': 'WEBP', '.tiff': 'TIFF'}
                save_fmt = fmt_map.get(suffix, 'JPEG')
                img_rgb.save(str(filepath), format=save_fmt)
            else:
                img_rgb = img.copy()

            # Hitung rata-rata kecerahan (grayscale)
            gray = img_rgb.convert('L')
            arr  = np.array(gray, dtype=np.float32)
            mean_brightness = float(arr.mean())
            result['rata_kecerahan'] = round(mean_brightness, 2)

            # Flag kecerahan — tidak dibuang, hanya ditandai
            if mean_brightness < dark_thresh:
                result['terindikasi_gelap'] = True
            elif mean_brightness > bright_thresh:
                result['terindikasi_overexposed'] = True

    except Exception as e:
        result['status'] = 'corrupt'
        result['error']  = str(e)

    return result


print("[INFO] Menganalisis kualitas gambar (ini bisa memakan waktu)...")
quality_records = []
for rec in tqdm(kept_records, desc="Filter kualitas", unit="img"):
    q = analyze_image_quality(rec['path'], DARK_THRESHOLD, BRIGHT_THRESHOLD)
    quality_records.append({
        'rel_path': rec['rel_path'],
        'label': rec['label'],
        'filename': rec['filename'],
        **q,
    })

print("[OK] Analisis kualitas selesai.")

[INFO] Menganalisis kualitas gambar (ini bisa memakan waktu)...


Filter kualitas:   0%|          | 0/30325 [00:00<?, ?img/s]

Filter kualitas:   0%|          | 103/30325 [00:00<00:29, 1024.40img/s]

Filter kualitas:   1%|          | 206/30325 [00:00<00:40, 740.58img/s] 

Filter kualitas:   1%|          | 285/30325 [00:00<00:42, 699.82img/s]

...

Filter kualitas:  99%|█████████▉| 30129/30325 [00:25<00:00, 605.32img/s] 

Filter kualitas: 100%|█████████▉| 30230/30325 [00:25<00:00, 514.56img/s]

Filter kualitas: 100%|█████████▉| 30311/30325 [00:25<00:00, 484.36img/s]

Filter kualitas: 100%|██████████| 30325/30325 [00:25<00:00, 1182.00img/s]

[OK] Analisis kualitas selesai.


In [10]:
df_quality = pd.DataFrame(quality_records)

# Gambar yang lulus filter (status ok)
df_clean = df_quality[df_quality['status'] == 'ok'].copy()
df_dropped = df_quality[df_quality['status'] != 'ok'].copy()

print("=== LAPORAN FILTER KUALITAS ===")
print(f"  Total sebelum filter : {len(df_quality)}")
print(f"  Status 'ok'          : {len(df_clean)}")
print(f"  Corrupt              : {(df_quality['status'] == 'corrupt').sum()}")
print(f"  Resolusi terlalu kecil: {(df_quality['status'] == 'dropped_low_res').sum()}")
print(f"  Dikonversi ke RGB    : {df_quality['dikonversi_ke_rgb'].sum()}")
print(f"  Terindikasi gelap    : {df_clean['terindikasi_gelap'].sum()}")
print(f"  Terindikasi overexposed: {df_clean['terindikasi_overexposed'].sum()}")

# Simpan metadata kualitas
metadata_path = os.path.join(OUTPUT_DIR, "metadata_kualitas.csv")
df_quality.to_csv(metadata_path, index=False)
print(f"\n[OK] metadata_kualitas.csv disimpan: {metadata_path}")

# Per kelas
print("\n=== DISTRIBUSI KELAS SETELAH FILTER ===")
for label, cnt in sorted(df_clean['label'].value_counts().items()):
    print(f"  {label}: {cnt}")

=== LAPORAN FILTER KUALITAS ===
  Total sebelum filter : 30325
  Status 'ok'          : 30154
  Corrupt              : 0
  Resolusi terlalu kecil: 171
  Dikonversi ke RGB    : 0
  Terindikasi gelap    : 198
  Terindikasi overexposed: 4900

[OK] metadata_kualitas.csv disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\metadata_kualitas.csv

=== DISTRIBUSI KELAS SETELAH FILTER ===
  B3: 3466
  Organik: 13744
  Recylable: 12944


---
## TAHAP 3 – Class Weights & Rencana Augmentasi

In [11]:
# ============================================================
# HITUNG CLASS WEIGHTS (inversely proportional)
# Formula: weight_c = N_total / (N_classes * N_c)
# Ini adalah formula sklearn class_weight='balanced'
# ============================================================
class_counts = df_clean['label'].value_counts().to_dict()
n_total      = sum(class_counts.values())
n_classes    = len(class_counts)

class_weights = {
    label: round(n_total / (n_classes * count), 6)
    for label, count in class_counts.items()
}

# Simpan sebagai JSON
cw_path = os.path.join(OUTPUT_DIR, "class_weights.json")
with open(cw_path, 'w') as f:
    json.dump(class_weights, f, indent=2)

print("=== CLASS WEIGHTS ===")
for label, w in sorted(class_weights.items()):
    print(f"  {label}: count={class_counts[label]}, weight={w:.4f}")
print(f"\n[OK] class_weights.json disimpan: {cw_path}")

=== CLASS WEIGHTS ===
  B3: count=3466, weight=2.9000
  Organik: count=13744, weight=0.7313
  Recylable: count=12944, weight=0.7765

[OK] class_weights.json disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\class_weights.json


In [12]:
# ============================================================
# RENCANA AUGMENTASI (dicatat di sini, dieksekusi di Colab)
# ============================================================

# CATATAN AUGMENTASI UNTUK COLAB (Bagian B, Tahap 5):
#
# === Augmentasi UMUM (semua kelas, split TRAIN saja) ===
# - RandomHorizontalFlip(p=0.5)
# - RandomVerticalFlip(p=0.2)
# - ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05)
# - RandomResizedCrop(size=384, scale=(0.8, 1.0))  → sebelum SiglipImageProcessor
#
# === Augmentasi KHUSUS B3 (lebih agresif, untuk counter imbalance) ===
# - RandomHorizontalFlip(p=0.5)
# - RandomVerticalFlip(p=0.3)
# - RandomRotation(degrees=30)
# - ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
# - RandomResizedCrop(size=384, scale=(0.7, 1.0))
# - RandomGrayscale(p=0.05)
#
# PENTING: Augmentasi dilakukan SEBELUM SiglipImageProcessor
# (SiglipImageProcessor menangani resize ke 384x384 dan normalisasi ke [-1, 1])
# Augmentasi hanya aktif pada split TRAIN, tidak untuk val/test.
#
# File konfigurasi augmentasi disimpan terpisah agar bisa dibaca di Colab:

aug_config = {
    "note": "Augmentasi dieksekusi on-the-fly di DataLoader Colab, bukan di preprocessing lokal.",
    "apply_only_on_split": "train",
    "augmentasi_umum": [
        {"transform": "RandomHorizontalFlip", "p": 0.5},
        {"transform": "RandomVerticalFlip", "p": 0.2},
        {"transform": "ColorJitter", "brightness": 0.2, "contrast": 0.2, "saturation": 0.1, "hue": 0.05},
        {"transform": "RandomResizedCrop", "size": 384, "scale": [0.8, 1.0]},
    ],
    "augmentasi_B3_ekstra": [
        {"transform": "RandomHorizontalFlip", "p": 0.5},
        {"transform": "RandomVerticalFlip", "p": 0.3},
        {"transform": "RandomRotation", "degrees": 30},
        {"transform": "ColorJitter", "brightness": 0.4, "contrast": 0.4, "saturation": 0.2, "hue": 0.1},
        {"transform": "RandomResizedCrop", "size": 384, "scale": [0.7, 1.0]},
        {"transform": "RandomGrayscale", "p": 0.05},
    ],
    "urutan_pipeline": [
        "1. Load gambar PIL",
        "2. Terapkan augmentasi (jika split=train)",
        "3. SiglipImageProcessor (resize 384x384, normalize ke [-1,1])",
        "4. Output tensor pixel_values untuk model"
    ]
}

aug_path = os.path.join(OUTPUT_DIR, "augmentasi_config.json")
with open(aug_path, 'w', encoding='utf-8') as f:
    json.dump(aug_config, f, indent=2, ensure_ascii=False)
print(f"[OK] augmentasi_config.json disimpan: {aug_path}")

[OK] augmentasi_config.json disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\augmentasi_config.json


---
## TAHAP 4 – Stratified Split Train/Val/Test

In [13]:
# Gunakan hanya file dengan status 'ok' dari Tahap 2
# rel_path adalah path relatif terhadap DATASET_DIR → portable ke Google Drive

np.random.seed(RANDOM_SEED)

df_split_input = df_clean[['rel_path', 'label']].copy().reset_index(drop=True)

# Split 1: train vs (val + test)
df_train, df_temp = train_test_split(
    df_split_input,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_split_input['label'],
    random_state=RANDOM_SEED,
)

# Split 2: val vs test dari sisa
val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)  # proporsi val dari temp
df_val, df_test = train_test_split(
    df_temp,
    test_size=(1 - val_frac),
    stratify=df_temp['label'],
    random_state=RANDOM_SEED,
)

# Verifikasi tidak ada overlap
train_paths = set(df_train['rel_path'])
val_paths   = set(df_val['rel_path'])
test_paths  = set(df_test['rel_path'])
assert len(train_paths & val_paths) == 0, "ADA OVERLAP train-val!"
assert len(train_paths & test_paths) == 0, "ADA OVERLAP train-test!"
assert len(val_paths & test_paths) == 0, "ADA OVERLAP val-test!"

print("=== HASIL SPLIT ===")
print(f"  Train : {len(df_train)} ({len(df_train)/len(df_split_input)*100:.1f}%)")
print(f"  Val   : {len(df_val)} ({len(df_val)/len(df_split_input)*100:.1f}%)")
print(f"  Test  : {len(df_test)} ({len(df_test)/len(df_split_input)*100:.1f}%)")
print()

for split_name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f"  {split_name} per kelas:")
    for label, cnt in sorted(df_s['label'].value_counts().items()):
        print(f"    {label}: {cnt}")

=== HASIL SPLIT ===
  Train : 21107 (70.0%)
  Val   : 4523 (15.0%)
  Test  : 4524 (15.0%)

  Train per kelas:
    B3: 2426
    Organik: 9620
    Recylable: 9061
  Val per kelas:
    B3: 520
    Organik: 2062
    Recylable: 1941
  Test per kelas:
    B3: 520
    Organik: 2062
    Recylable: 1942


In [14]:
# Simpan CSV split
# Format: rel_path (relatif terhadap DATASET_DIR), label
# Path menggunakan forward slash agar portabel ke Linux/Colab

def normalize_path_sep(df: pd.DataFrame) -> pd.DataFrame:
    """Ganti backslash dengan forward slash untuk portabilitas."""
    df = df.copy()
    df['rel_path'] = df['rel_path'].str.replace('\\', '/', regex=False)
    return df

df_train_out = normalize_path_sep(df_train)
df_val_out   = normalize_path_sep(df_val)
df_test_out  = normalize_path_sep(df_test)

train_csv = os.path.join(OUTPUT_DIR, "train.csv")
val_csv   = os.path.join(OUTPUT_DIR, "val.csv")
test_csv  = os.path.join(OUTPUT_DIR, "test.csv")

df_train_out.to_csv(train_csv, index=False)
df_val_out.to_csv(val_csv, index=False)
df_test_out.to_csv(test_csv, index=False)

print(f"[OK] train.csv: {train_csv}")
print(f"[OK] val.csv  : {val_csv}")
print(f"[OK] test.csv : {test_csv}")

[OK] train.csv: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\train.csv
[OK] val.csv  : d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\val.csv
[OK] test.csv : d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\test.csv


---
## RINGKASAN AKHIR & INSTRUKSI UPLOAD KE GOOGLE DRIVE

In [15]:
print("="*60)
print("  PREPROCESSING LOKAL SELESAI")
print("="*60)
print()
print(f"Output disimpan di: {OUTPUT_DIR}")
print()
print("File yang dihasilkan:")
for fname in ["dedup_log.csv", "metadata_kualitas.csv", "class_weights.json",
              "augmentasi_config.json", "train.csv", "val.csv", "test.csv"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  [{size:>8,} bytes] {fname}")

print()
print("="*60)
print("  INSTRUKSI UPLOAD KE GOOGLE DRIVE")
print("="*60)
print("""
Yang perlu diunggah ke Google Drive:

  1. Folder Dataset (TANPA subfolder duplicates_removed)
     → Folder: Dataset/0_Recylable, Dataset/1_Organik, Dataset/2_B3
     → SKIP: Dataset/duplicates_removed (tidak perlu)

  2. Folder preprocessing_output (seluruh isinya):
     → train.csv
     → val.csv
     → test.csv
     → class_weights.json
     → augmentasi_config.json
     → metadata_kualitas.csv  (opsional, untuk referensi)
     → dedup_log.csv          (opsional, untuk referensi)

Struktur yang direkomendasikan di Google Drive:
  MyDrive/
  └── Senpro/
      ├── Dataset/
      │   ├── 0_Recylable/
      │   ├── 1_Organik/
      │   └── 2_B3/
      └── preprocessing_output/
          ├── train.csv
          ├── val.csv
          ├── test.csv
          ├── class_weights.json
          └── augmentasi_config.json

Di notebook Colab, set:
  ROOT_DIR = "/content/drive/MyDrive/Senpro"
""")

  PREPROCESSING LOKAL SELESAI

Output disimpan di: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output

File yang dihasilkan:
  [       2 bytes] dedup_log.csv
  [2,573,104 bytes] metadata_kualitas.csv
  [      72 bytes] class_weights.json
  [   1,415 bytes] augmentasi_config.json
  [ 673,167 bytes] train.csv
  [ 144,159 bytes] val.csv
  [ 144,273 bytes] test.csv

  INSTRUKSI UPLOAD KE GOOGLE DRIVE

Yang perlu diunggah ke Google Drive:

  1. Folder Dataset (TANPA subfolder duplicates_removed)
     → Folder: Dataset/0_Recylable, Dataset/1_Organik, Dataset/2_B3
     → SKIP: Dataset/duplicates_removed (tidak perlu)

  2. Folder preprocessing_output (seluruh isinya):
     → train.csv
     → val.csv
     → test.csv
     → class_weights.json
     → augmentasi_config.json
     → metadata_kualitas.csv  (opsional, untuk referensi)
     → dedup_log.csv          (opsional, untuk referensi)

Struktur yang direkomendasikan di Google Drive:
  MyDrive/
  └── Senpro/
      ├── Dataset/
      │